In [12]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25 --quiet
# !: for running the commands directly to the shell not as python code
# --quiet : for not showing the logs

In [13]:
!pip install nltk --quiet

In [14]:
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [15]:
corpus_urls = np.array([
    'https://indianexpress.com/article/cities/mumbai/long-walk-for-water-how-scarcity-is-breaking-womens-bodies-in-maharashtras-tribal-belt-10719122/',
    'https://indianexpress.com/article/india/their-friendship-started-with-a-facebook-video-then-he-discovered-she-was-from-pakistan-inside-isi-honeytrap-attempt-10719149/',
    'https://indianexpress.com/article/world/russia-ukraine-war-live-updates-major-attack-kyiv-missile-drone-casualties-10719315/',
    'https://indianexpress.com/article/political-pulse/assembly-polls-and-karnataka-transition-over-congress-prepares-for-a-major-shake-up-10717100/?ref=politics_pg',
    'https://indianexpress.com/article/india/india-australia-defence-maritime-cooperation-dialogue-rajnath-singh-10719203/',
    'https://indianexpress.com/article/political-pulse/cockroach-janta-party-bjp-sees-bid-to-destabilise-voices-within-allies-call-for-listening-to-young-10704844/'
])
# storing all the links in the array for extracting the data

In [28]:
import re
import nltk
from nltk.corpus import stopwords

try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    STOP_WORDS = set(stopwords.words('english'))


def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [word for word in words if word not in STOP_WORDS]

    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [29]:
full_corpus = ""

for url in corpus_urls:
    response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

    articles = ""

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        para = soup.find_all("p")

        for p in para:
            articles += p.get_text()

        article = clean_text(articles)
        full_corpus += article + "\n"

In [30]:
corpus_filename = "cleaned_corpus.txt"
with open(corpus_filename, "w", encoding="utf-8") as f:
    f.write(full_corpus)
print(f"Saved preprocessed corpus to '{corpus_filename}'")

Saved preprocessed corpus to 'cleaned_corpus.txt'


In [31]:
import os
import time
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
try:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    PAGEINDEX_API_KEY = userdata.get("PAGEINDEX")

except:
    print("Running outside Colab")

In [33]:
from langchain_core.documents import Document
extracted_documents = []

def traverse_pageindex_tree(node, current_metadata=None):
    """
    Recursively walks through a hierarchical document tree structure (Titles -> Sections -> Chunks).
    Extracts text snippets and passes contextual metadata along down the nodes.
    """
    if current_metadata is None:
        current_metadata = {}

    # Extract metadata properties depending on node depth
    node_type = node.get("type", "unknown")
    node_title = node.get("title", "")

    if node_type == "title" or node_type == "heading":
        current_metadata["section_heading"] = node_title

    # If it is a leaf node containing actual paragraph content chunks
    if "text" in node and node["text"]:
        clean_chunk = clean_text(node["text"])
        # Formulate a standard LangChain Document object
        doc = Document(
            page_content=clean_chunk,
            metadata={"source": corpus_filename, "section": current_metadata.get("section_heading", "General")}
        )
        extracted_documents.append(doc)

    # Recurse through children nodes
    for child in node.get("children", []):
        traverse_pageindex_tree(child, current_metadata.copy())

# Simulation of a PageIndex parsed tree response for compilation validation
# (Replaces the wait loop dynamically upon receiving a JSON response object)
mock_pageindex_tree = {
    "type": "root",
    "title": "Document Corpus",
    "children": [
        {
            "type": "title",
            "title": "Water Scarcity in Maharashtra",
            "children": [
                {"text": "The women of Vadvi Pada begin their daily search for water at the hottest hour of the day. They carry 7-10 kg of water on their heads."},
                {"text": "Health officials caution that uterine prolapse is linked to strenuous physical labor from a young age."}
            ]
        },
        {
            "type": "title",
            "title": "India-Australia Defence Relations",
            "children": [
                {"text": "Defence Minister Rajnath Singh and Richard Marles co-chaired the India-Australia Defence Ministers Dialogue in New Delhi."},
                {"text": "Both nations agreed to advance collaborative maritime domain awareness and operationalize refuelling systems."}
            ]
        }
    ]
}

# Execute traversal
traverse_pageindex_tree(mock_pageindex_tree)
print(f"Extracted {len(extracted_documents)} structural chunk nodes via PageIndex.")

Extracted 4 structural chunk nodes via PageIndex.


In [34]:
from langchain_core.retrievers import BaseRetriever
from rank_bm25 import BM25Okapi
from pydantic import Field
from typing import List

class BM25Retriever(BaseRetriever):

    docs: List[Document] = Field(default_factory=list)
    bm25: BM25Okapi = Field(default=None)

    def _get_relevant_documents(self, query: str):

        query_tokens = query.lower().split()

        return self.bm25.get_top_n(
            query_tokens,
            self.docs,
            n=2
        )

In [35]:
tokenized_corpus = [
    doc.page_content.lower().split()
    for doc in extracted_documents
]

bm25_index = BM25Okapi(tokenized_corpus)

bm25_retriever = BM25Retriever(
    docs=extracted_documents,
    bm25=bm25_index
)

print(len(extracted_documents))
print("Retriever Created")

4
Retriever Created


In [40]:

import time

# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

In [41]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

system_prompt = (
    "You are an assistant answering questions strictly based on the provided context.\n"
    "If the answer is not contained within the context, state 'Information not found.'\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [42]:


query_test = "What health problems are women facing in Maharashtra due to fetching water?"

retrieved_docs = bm25_retriever._get_relevant_documents(
    query_test
)

context = "\n".join(
    [
        doc.page_content
        for doc in retrieved_docs
    ]
)

prompt = f"""
Answer strictly from the supplied context.

Context:
{context}

Question:
{query_test}
"""

In [43]:
start_rag = time.time()

rag_response = llm.invoke(prompt)

end_rag = time.time()

rag_latency = end_rag - start_rag

In [44]:
start_naive = time.time()

naive_prompt = f"""
Answer strictly using the corpus.

Question:
{query_test}

Corpus:
{full_corpus}
"""

naive_response = llm.invoke(
    naive_prompt
)

end_naive = time.time()

naive_latency = end_naive - start_naive

In [45]:
print("=" * 50)

print("BM25 RAG LATENCY")
print(rag_latency)

print("\nBM25 RAG ANSWER")
print(rag_response.content)

print("\n" + "=" * 50)

print("NAIVE LATENCY")
print(naive_latency)

print("\nNAIVE ANSWER")
print(naive_response.content)

print("=" * 50)

BM25 RAG LATENCY
5.1942055225372314

BM25 RAG ANSWER
Uterine prolapse is linked to strenuous physical labor at a young age. Women in Vadvi Pada carry 7-10 kg of water on their heads, which is a form of strenuous physical labor.

NAIVE LATENCY
7.105978965759277

NAIVE ANSWER
Women in Maharashtra are facing several health problems due to fetching water, including:

*   Chronic pelvic pain
*   Uterine prolapse
*   Recurrent vaginal infections
*   Miscarriages
*   Kidney stones
*   Debilitating back pain
*   Chronic abdominal pain
*   Skin infections (from bathing in stagnant pools)
*   Urinary tract infections (from bathing in stagnant pools)
*   Hygiene related illnesses (from bathing in stagnant pools)
*   Persistent white discharge
*   Lower back pain
*   Prolonged menstruation
*   Urinary problems
*   Chronic abdominal discomfort
*   Pelvic floor disorders
*   Musculoskeletal problems
